# RAG caseiro: chunking, busca hibrida, geracao com citacao e testes de falha

Este notebook implementa um pipeline RAG completo, do zero, sem framework (sem LangChain/LlamaIndex) e sem banco vetorial externo — so para deixar claro exatamente o que cada peca faz.

**Documentos de exemplo**: politicas internas de uma empresa ficticia ("Aurora Tech"), escritas para este notebook. Troque pelos seus proprios documentos (PDFs convertidos em texto, suas anotacoes, etc.) na secao 1.

**Sobre embeddings**: a secao principal usa **TF-IDF** (via scikit-learn) como um "embedding" simples e totalmente offline, sem precisar de chave de API — isso deixa o notebook 100% executavel do jeito que esta. A secao 7 mostra como trocar por **embeddings densos reais** (OpenAI, Voyage AI) via API, que capturam similaridade semantica de verdade (TF-IDF so captura sobreposicao de palavras — a secao de testes de falha deixa essa limitacao bem clara).

**Requer para a secao de geracao**: `pip install anthropic` e uma `ANTHROPIC_API_KEY`.

## 1. Documentos e chunking

In [ ]:
documents = [
    {
        "id": "politica_ferias",
        "text": """Politica de ferias da empresa Aurora Tech.
Todo colaborador tem direito a 30 dias corridos de ferias por ano trabalhado.
As ferias devem ser solicitadas com no minimo 30 dias de antecedencia ao gestor direto.
E permitido o fracionamento das ferias em ate 3 periodos, sendo que um deles nao pode ser inferior a 14 dias corridos.
Colaboradores com menos de 12 meses de empresa nao tem direito a ferias proporcionais antes do primeiro periodo aquisitivo completo."""
    },
    {
        "id": "politica_reembolso",
        "text": """Politica de reembolso de despesas da Aurora Tech.
Despesas com viagens a trabalho devem ser reembolsadas em ate 15 dias uteis apos o envio da nota fiscal.
O limite diario para alimentacao em viagem e de R$ 120,00.
Despesas acima de R$ 500,00 exigem aprovacao previa do gestor antes da viagem.
Notas fiscais devem ser enviadas atraves do sistema interno ate 5 dias uteis apos o retorno da viagem."""
    },
    {
        "id": "politica_home_office",
        "text": """Politica de trabalho remoto da Aurora Tech.
Colaboradores dos times de engenharia e produto podem trabalhar em regime hibrido, com minimo de 2 dias presenciais por semana.
O auxilio home office e de R$ 150,00 mensais, pago junto com o salario.
Colaboradores em regime 100% remoto precisam de aprovacao especifica da diretoria.
O horario nucleo, em que todos devem estar disponiveis, e das 10h as 16h."""
    },
    {
        "id": "politica_contratacao",
        "text": """Processo de contratacao da Aurora Tech.
Todo processo seletivo passa por 3 etapas: triagem de curriculo, entrevista tecnica e entrevista com a lideranca.
O prazo medio entre a primeira entrevista e a proposta final e de 3 semanas.
Propostas de emprego sao validas por 5 dias uteis a partir do envio.
Colaboradores contratados passam por um periodo de experiencia de 90 dias."""
    },
    {
        "id": "politica_desligamento",
        "text": """Politica de desligamento da Aurora Tech.
Em caso de desligamento sem justa causa, o aviso previo e de 30 dias, podendo ser trabalhado ou indenizado.
Ferias vencidas e proporcionais sao pagas na rescisao, com acrescimo de um terco constitucional.
O acesso aos sistemas internos e revogado no ultimo dia util do colaborador.
Colaboradores desligados podem solicitar carta de recomendacao ao gestor direto em ate 90 dias apos o desligamento."""
    },
]

print(f"{len(documents)} documentos carregados")


Chunking por palavras com overlap: cada chunk tem `chunk_size` palavras, e o proximo chunk comeca `overlap` palavras antes do fim do anterior — isso evita que uma informacao seja cortada exatamente na fronteira entre dois chunks sem aparecer inteira em nenhum deles.

In [ ]:
def chunk_text(text, chunk_size=40, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks


all_chunks = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({"doc_id": doc["id"], "chunk_id": f"{doc['id']}_{i}", "text": chunk})

print(f"Total de chunks: {len(all_chunks)}")
for c in all_chunks[:3]:
    print(c["chunk_id"], "->", c["text"][:60], "...")


## 2. Indexacao com TF-IDF (stand-in offline para embeddings densos)

TF-IDF representa cada chunk como um vetor esparso baseado na frequencia das palavras, ponderada pela raridade delas no corpus. Nao captura sinonimos nem significado (por isso os testes de falha na secao 5 vao expor essa limitacao de proposito), mas e rapido, gratuito e ilustra bem o mecanismo de "buscar por similaridade de vetores" antes de trocar por embeddings semanticos de verdade.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

chunk_texts = [c["text"] for c in all_chunks]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

def search_tfidf(query, top_k=3):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(all_chunks[i], float(sims[i])) for i in top_idx]


## 3. Busca por palavra-chave: BM25 implementado a mao

BM25 e o algoritmo classico de busca por relevancia de termos (usado por motores de busca ha decadas, antes de embeddings existirem). Ele captura bem casos onde a correspondencia exata de uma palavra importa mais que "significado geral" — por exemplo, buscar por um termo tecnico especifico ou um numero.

In [ ]:
import re
import math
from collections import Counter

def tokenize(text):
    return re.findall(r"\w+", text.lower())

class SimpleBM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(d) for d in documents]
        self.doc_lens = [len(d) for d in self.docs]
        self.avg_len = sum(self.doc_lens) / len(self.doc_lens)
        self.doc_freqs = [Counter(d) for d in self.docs]
        self.n_docs = len(documents)
        self.idf = self._compute_idf()

    def _compute_idf(self):
        df = Counter()
        for doc in self.docs:
            for term in set(doc):
                df[term] += 1
        return {
            term: math.log((self.n_docs - freq + 0.5) / (freq + 0.5) + 1)
            for term, freq in df.items()
        }

    def score(self, query, idx):
        q_terms = tokenize(query)
        doc_freq = self.doc_freqs[idx]
        doc_len = self.doc_lens[idx]
        score = 0.0
        for term in q_terms:
            if term not in doc_freq:
                continue
            idf = self.idf.get(term, 0.0)
            freq = doc_freq[term]
            numer = freq * (self.k1 + 1)
            denom = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avg_len)
            score += idf * numer / denom
        return score

    def search(self, query, top_k=3):
        scores = [self.score(query, i) for i in range(self.n_docs)]
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(all_chunks[i], scores[i]) for i in top_idx]


bm25 = SimpleBM25(chunk_texts)


## 4. Busca hibrida: Reciprocal Rank Fusion (RRF)

Em vez de somar os scores brutos de TF-IDF e BM25 diretamente (eles estao em escalas diferentes e nao sao comparaveis), o RRF combina os **rankings** de cada metodo: cada documento ganha pontos com base na sua *posicao* em cada lista, nao no valor bruto do score. E simples e funciona bem na pratica.

In [ ]:
def reciprocal_rank_fusion(rankings_list, k=60):
    scores = Counter()
    for ranking in rankings_list:
        for rank, chunk_id in enumerate(ranking):
            scores[chunk_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def search_hybrid(query, top_k=3):
    tfidf_results = search_tfidf(query, top_k=len(all_chunks))
    bm25_results = bm25.search(query, top_k=len(all_chunks))

    tfidf_ranking = [c["chunk_id"] for c, _ in tfidf_results]
    bm25_ranking = [c["chunk_id"] for c, _ in bm25_results]

    fused = reciprocal_rank_fusion([tfidf_ranking, bm25_ranking])
    chunk_by_id = {c["chunk_id"]: c for c in all_chunks}
    return [(chunk_by_id[cid], score) for cid, score in fused[:top_k]]


## 5. Testes deliberados: onde a busca funciona e onde falha

Isso e mais importante do que parece. Testar so os casos faceis da uma falsa sensacao de que o RAG "funciona" — os casos dificeis sao os que realmente importam em producao.

In [ ]:
def mostrar_resultados(titulo, results):
    print(f"\n--- {titulo} ---")
    for chunk, score in results:
        print(f"{score:.3f} | {chunk['chunk_id']} | {chunk['text'][:70]}...")


# caso 1: vocabulario igual ao documento -> deve funcionar bem
mostrar_resultados(
    "Caso facil (vocabulario igual ao documento)",
    search_hybrid("quantos dias de ferias tem direito o colaborador?")
)

# caso 2: vocabulario DIFERENTE do documento -> expõe a limitacao de TF-IDF/BM25
# ("descanso remunerado dividido em partes" == "ferias fracionadas", mas nenhuma palavra bate)
mostrar_resultados(
    "Caso dificil: sinonimos (TF-IDF/BM25 nao entendem 'descanso' = 'ferias')",
    search_hybrid("posso tirar descanso remunerado dividido em partes do ano?")
)

# caso 3: pergunta que exige agregar informacao de VARIOS documentos
# nenhum chunk isolado responde -- a resposta esta espalhada em 3 documentos diferentes
mostrar_resultados(
    "Caso dificil: agregacao multi-documento (nenhum chunk isolado responde)",
    search_hybrid("quais politicas mencionam prazos contados em dias uteis?")
)


Repare no segundo caso: como nem TF-IDF nem BM25 entendem que "descanso remunerado dividido em partes" significa a mesma coisa que "ferias fracionadas", a busca retorna chunks fracos ou errados. **Isso e exatamente o tipo de falha que embeddings densos (semanticos) resolvem melhor** — veja a secao 7.

No terceiro caso, a informacao esta espalhada em 3 documentos diferentes (ferias, reembolso, contratacao todos mencionam prazos em dias uteis). Nenhum chunk isolado contem a resposta completa — isso e uma limitacao estrutural do RAG "ingenuo": ele recupera *trechos*, nao *agrega e raciocina* sobre o corpus inteiro.

## 6. Geracao com citacao obrigatoria

Depois de recuperar os chunks, montamos um prompt que exige que o modelo cite explicitamente de qual chunk tirou cada informacao — isso torna possivel auditar se a resposta esta realmente fundamentada no que foi recuperado, ou se o modelo "alucinou" por cima.

In [ ]:
import os
from anthropic import Anthropic

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def montar_prompt(pergunta, chunks_recuperados):
    contexto = "\n\n".join(
        f"[{c['chunk_id']}]\n{c['text']}" for c, _ in chunks_recuperados
    )
    return f"""Responda a pergunta usando APENAS as informacoes fornecidas abaixo. Para cada afirmacao, cite o id do chunk entre colchetes (ex: [politica_ferias_0]). Se a informacao nao estiver disponivel nos chunks, diga explicitamente que nao encontrou a resposta -- nao invente.

<contexto>
{contexto}
</contexto>

Pergunta: {pergunta}"""


def responder(pergunta, top_k=3):
    chunks_recuperados = search_hybrid(pergunta, top_k=top_k)
    prompt = montar_prompt(pergunta, chunks_recuperados)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text, chunks_recuperados


resposta, chunks_usados = responder("quantos dias de ferias tem direito o colaborador?")
print(resposta)


## 7. Trocando por embeddings densos reais (semanticos)

TF-IDF/BM25 so capturam sobreposicao de palavras. Para resolver o "caso dificil de sinonimos" da secao 5, o proximo passo e usar um **modelo de embedding denso** — treinado para colocar textos com significado parecido proximos no espaco vetorial, mesmo sem palavras em comum.

O codigo abaixo mostra como plugar a API de embeddings da OpenAI no lugar do TF-IDF (a logica de busca por cosseno e a mesma). Requer `pip install openai` e uma chave de API da OpenAI.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()  # usa OPENAI_API_KEY do ambiente

def embed_texts(texts, model="text-embedding-3-small"):
    response = openai_client.embeddings.create(model=model, input=texts)
    return np.array([item.embedding for item in response.data])


# gera embeddings densos para todos os chunks (uma vez so, custo baixo)
dense_embeddings = embed_texts(chunk_texts)

def search_dense(query, top_k=3):
    q_emb = embed_texts([query])
    sims = cosine_similarity(q_emb, dense_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(all_chunks[i], float(sims[i])) for i in top_idx]


# repetir o "caso dificil de sinonimos" com embeddings densos
mostrar_resultados(
    "Caso dificil de sinonimos, agora com embeddings densos",
    search_dense("posso tirar descanso remunerado dividido em partes do ano?")
)
# expectativa: embeddings densos devem recuperar o chunk de ferias corretamente,
# porque "descanso remunerado" e "ferias" ficam proximos no espaco vetorial
# mesmo sem nenhuma palavra em comum -- diferente de TF-IDF/BM25.


## 8. Bonus: re-ranking com LLM

Um re-ranker pega os top-k candidatos de uma busca rapida (TF-IDF, BM25 ou embeddings densos) e usa um modelo mais lento e mais preciso para reavaliar a relevancia de cada um antes de decidir quais realmente entram no prompt final. Aqui simulamos isso pedindo ao proprio Claude para pontuar a relevancia de cada chunk recuperado.

In [ ]:
def rerank_com_llm(pergunta, chunks_candidatos):
    scored = []
    for chunk, _ in chunks_candidatos:
        prompt = f"""Numa escala de 0 a 10, quao relevante e o trecho abaixo para responder a pergunta? Responda APENAS com o numero.

Pergunta: {pergunta}
Trecho: {chunk['text']}"""
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=10,
            messages=[{"role": "user", "content": prompt}],
        )
        try:
            score = float(response.content[0].text.strip())
        except ValueError:
            score = 0.0
        scored.append((chunk, score))

    return sorted(scored, key=lambda x: x[1], reverse=True)


candidatos = search_hybrid("posso tirar descanso remunerado dividido em partes do ano?", top_k=5)
reranked = rerank_com_llm("posso tirar descanso remunerado dividido em partes do ano?", candidatos)
mostrar_resultados("Apos re-ranking com LLM", reranked)


## Notas finais e proximos passos

- **Banco vetorial de verdade**: aqui usamos listas em memoria, que so funcionam para poucos documentos. Para escalar, troque por Chroma (`pip install chromadb`, roda local) ou um servico gerenciado como Pinecone
- **Chunking mais sofisticado**: experimente chunking por paragrafo/secao (respeitando a estrutura do documento) em vez de tamanho fixo em palavras, especialmente se seus documentos tiverem titulos e secoes claras
- **A licao mais importante desta fase**: o "caso dificil de sinonimos" e o "caso de agregacao multi-documento" nao sao bugs de implementacao — sao limitacoes estruturais de diferentes abordagens de RAG. Embeddings densos resolvem o primeiro; o segundo geralmente exige uma abordagem diferente (ex: sumarizacao previa dos documentos, ou dar ao agente uma ferramenta de busca que ele pode chamar varias vezes com reformulacoes da pergunta — o que conecta diretamente com a Fase 5)
- **Proximo passo natural**: na Fase 5, RAG deixa de ser um pipeline fixo e vira uma **ferramenta que um agente decide quando e como chamar** — inclusive podendo reformular a pergunta e buscar de novo se a primeira tentativa nao trouxe informacao suficiente
